# Duke Bites — Evaluation Notebook
This notebook covers:
1. Setup
2. Prompt engineering comparison (3 variants)
3. Retrieval quality evaluation
4. Multi-turn conversation test
5. Edge case analysis
6. Results summary table

## 1. Setup

In [ ]:
!pip install sentence-transformers groq --quiet

from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.chdir('/content/drive/MyDrive/cs372_final_project')
sys.path.append('src')
os.environ['GROQ_API_KEY'] = 'gsk_your_key_here'

from groq import Groq
from retrieval import retrieve, format_context
from prompts import PROMPT_V1_FRIENDLY, PROMPT_V2_CONCISE, PROMPT_V3_NUTRITIONIST

client = Groq(api_key=os.environ['GROQ_API_KEY'])
MODEL  = 'llama-3.3-70b-versatile'
print('Setup complete.')

## 2. Prompt Engineering Comparison
We test 3 system prompt variants on the same queries and score each response.
This satisfies the rubric item: *'Applied prompt engineering with evaluation of at least three prompt designs'* (3 pts)

In [ ]:
def ask(prompt_template, user_message):
    results = retrieve(user_message, top_k=5)
    context = format_context(results)
    system  = prompt_template.format(context=context)
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user_message}
        ],
        temperature=0.7,
        max_tokens=300
    )
    return resp.choices[0].message.content

test_queries = [
    "I want something spicy for dinner",
    "I'm vegetarian and want a healthy lunch",
    "I need a late night snack",
]

prompts = {
    "V1 Friendly":     PROMPT_V1_FRIENDLY,
    "V2 Concise":      PROMPT_V2_CONCISE,
    "V3 Nutritionist": PROMPT_V3_NUTRITIONIST,
}

# Store all responses for the comparison table below
responses = {q: {} for q in test_queries}

for q in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: '{q}'")
    print('='*60)
    for name, prompt in prompts.items():
        reply = ask(prompt, q)
        responses[q][name] = reply
        print(f"\n[{name}]\n{reply}")

In [ ]:
# Prompt comparison table
# Score each response manually on 3 criteria (1-5 scale)
# Relevance: did it recommend food matching the query?
# Clarity:   was the response easy to read and understand?
# Helpfulness: did it include location, hours, useful context?

import pandas as pd

# Fill in your scores after reading the responses above
scores = {
    "Prompt Variant":  ["V1 Friendly", "V2 Concise", "V3 Nutritionist"],
    "Relevance (1-5)": [0, 0, 0],   # <- fill in
    "Clarity (1-5)":   [0, 0, 0],   # <- fill in
    "Helpfulness (1-5)":[0, 0, 0],  # <- fill in
    "Avg Score":       [0, 0, 0],   # <- fill in
    "Notes":           ["", "", ""] # <- fill in
}

scores_df = pd.DataFrame(scores)
print(scores_df.to_string(index=False))
print("\nWinner: V? — [write your reasoning here]")

## 3. Retrieval Quality Evaluation
Measures how well the embedding search returns relevant items.
Satisfies: *'Used sentence embeddings for semantic similarity or retrieval'* (5 pts) and *'Used at least three distinct evaluation metrics'* (3 pts)

In [ ]:
# Metric 1: Average similarity score — how confident is the retrieval?
# Metric 2: Tag overlap — do returned items' tags match the query intent?
# Metric 3: Manual precision@3 — of top 3 results, how many are actually relevant?

eval_queries = [
    {"query": "I want something spicy",          "expected_tags": ["spicy"]},
    {"query": "healthy vegan option",            "expected_tags": ["vegan", "healthy"]},
    {"query": "I want sushi",                    "expected_tags": ["sushi", "japanese"]},
    {"query": "comfort food for breakfast",      "expected_tags": ["breakfast", "comfort"]},
    {"query": "late night burger",               "expected_tags": ["burger", "late_night"]},
    {"query": "something light and fresh",       "expected_tags": ["light", "fresh", "salad"]},
    {"query": "indian food",                     "expected_tags": ["indian", "spicy"]},
    {"query": "I want a coffee or tea drink",    "expected_tags": ["caffeinated", "drink"]},
]

rows = []
for eq in eval_queries:
    results  = retrieve(eq["query"], top_k=3)
    avg_score = round(sum(r["score"] for r in results) / len(results), 3)

    # Tag overlap: fraction of expected tags found in any of the top 3 results
    all_tags = " ".join(r["tags"] for r in results).lower()
    matched  = sum(1 for t in eq["expected_tags"] if t in all_tags)
    tag_overlap = round(matched / len(eq["expected_tags"]), 2)

    rows.append({
        "Query":          eq["query"],
        "Avg Score":      avg_score,
        "Tag Overlap":    tag_overlap,
        "Top Result":     results[0]["item"] + " @ " + results[0]["location"],
    })

eval_df = pd.DataFrame(rows)
print(eval_df.to_string(index=False))
print(f"\nMean avg score:   {eval_df['Avg Score'].mean():.3f}")
print(f"Mean tag overlap: {eval_df['Tag Overlap'].mean():.2f}")

## 4. Multi-Turn Conversation Test
Verifies context carries across turns.
Satisfies: *'Built multi-turn conversation system with context management and history tracking'* (7 pts)

In [ ]:
from chatbot import chat

history = []
turns = [
    "I'm really hungry and want something filling",
    "Actually I'm vegetarian, anything like that?",
    "What time does that place close?",
    "Is there anything spicy in that style?",
]

for msg in turns:
    print(f"\nUser: {msg}")
    reply, history = chat(msg, history)
    print(f"Duke Bites: {reply}")

print(f"\nTotal turns in history: {len(history)}")
print("Multi-turn context working:", len(history) == len(turns) * 2)

## 5. Edge Case Analysis
Satisfies: *'Performed error analysis with visualization and discussion of failure cases'* (7 pts)
and *'Analyzed model behavior on edge cases or out-of-distribution examples'* (5 pts)

In [ ]:
edge_cases = [
    "I want a cheeseburger at 3am",          # out of hours
    "I'm allergic to gluten and dairy",       # multiple restrictions
    "xyzabc random gibberish food",           # nonsense query
    "I want the cheapest thing possible",     # no price data
    "Something from East Campus",             # location not in dataset
]

print("Edge Case Analysis")
print("=" * 60)
for q in edge_cases:
    results = retrieve(q, top_k=3)
    reply, _ = chat(q, [])
    print(f"\nQuery: '{q}'")
    print(f"Top retrieval score: {results[0]['score']}")
    print(f"Top result: {results[0]['item']} @ {results[0]['location']}")
    print(f"Response: {reply[:200]}...")
    print("-" * 40)

In [ ]:
# Visualize retrieval score distribution across edge cases vs normal queries
import matplotlib.pyplot as plt

normal_queries = [
    "I want something spicy",
    "healthy vegan lunch",
    "sushi for dinner",
    "breakfast sandwich",
    "late night burger",
]

normal_scores = [retrieve(q, top_k=1)[0]["score"] for q in normal_queries]
edge_scores   = [retrieve(q, top_k=1)[0]["score"] for q in edge_cases]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([f"N{i+1}" for i in range(len(normal_queries))], normal_scores,
       color="steelblue", label="Normal queries")
ax.bar([f"E{i+1}" for i in range(len(edge_cases))], edge_scores,
       color="tomato", label="Edge cases")
ax.axhline(y=0.3, color="gray", linestyle="--", label="Threshold (0.3)")
ax.set_ylabel("Top-1 Similarity Score")
ax.set_title("Retrieval Score: Normal Queries vs Edge Cases")
ax.legend()
plt.tight_layout()
plt.savefig("retrieval_scores.png", dpi=150)
plt.show()
print(f"Normal avg: {sum(normal_scores)/len(normal_scores):.3f}")
print(f"Edge avg:   {sum(edge_scores)/len(edge_scores):.3f}")

## 6. Results Summary
Satisfies: *'Conducted both qualitative and quantitative evaluation with thoughtful discussion'* (5 pts)

In [ ]:
summary = {
    "Metric": [
        "Best prompt variant",
        "Mean retrieval score (normal queries)",
        "Mean retrieval score (edge cases)",
        "Mean tag overlap",
        "Multi-turn context working",
        "Total menu items indexed",
        "Embedding model",
        "LLM",
    ],
    "Result": [
        "V? — [fill in after running section 2]",
        "[fill in from section 3]",
        "[fill in from section 5]",
        "[fill in from section 3]",
        "Yes",
        "394",
        "all-MiniLM-L6-v2",
        "Llama 3.3 70B via Groq",
    ]
}

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))